# Surrogate v2 — Plan + Knobs + Workload → Latency / Ranking (Improved)

Improvements over v1:
- **Phase 1**: `db_type` flag added from the start (fixes transfer dimension bug). Plan parse-failure flag added. Near-constant feature pruning.
- **Phase 2**: Log-transform target. Full GroupKFold CV with early stopping. Feature importance diagnostics.
- **Phase 3**: LambdaRank trained on ALL PG data with tuned hyperparams.
- **Phase 4**: Held-out MySQL test split. Pseudo-label quality gate. Importance-weighted PG rows. Two-stage fine-tuning. Includes experimental Residual Learning and Weighted Joint Training for regression transfer.
- **Cross-cutting**: Artifact versioning with config hash. Reproducibility guards.

Note: this notebook defaults to parsing structured plans (`PLAN_REPR=structured`). Set `PLAN_REPR=embedding` if you want to use precomputed plan embeddings in `qp_emb_vector` instead.

Phases:
- Phase 1: DB-agnostic representations (plan, knobs, workload) + domain flag
- Phase 2: PostgreSQL regression baseline — full CV, log-target, early stopping, diagnostics
- Phase 3: LambdaRank on PostgreSQL
- Phase 4: Transfer to MySQL — held-out split, pseudo-label gate, two-stage fine-tune, importance weighting
- Phase 5: Validate + save versioned artifacts

Uses repo CSVs:
- `surrogate/cost_model_collected.csv`
- `surrogate/cost_model_run_history.csv`

In [1]:
from __future__ import annotations

import ast
import hashlib
import json
import math
import os
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# ── Reproducibility ──────────────────────────────────────────────────────────
# LightGBM ignores np.random.seed; set PYTHONHASHSEED for full reproducibility.
os.environ.setdefault('PYTHONHASHSEED', '7')
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
def find_repo_root(start: Path) -> Path:
    start = start.resolve()

    # Case A: running from repo root (expects surrogate/...) or any subdir.
    for p in [start, *start.parents]:
        if (p / 'surrogate' / 'cost_model_collected.csv').exists() and (
            p / 'surrogate' / 'cost_model_run_history.csv'
        ).exists():
            return p

    # Case B: running from inside the surrogate dir.
    for p in [start, *start.parents]:
        if p.name == 'surrogate' and (p / 'cost_model_collected.csv').exists() and (
            p / 'cost_model_run_history.csv'
        ).exists():
            return p.parent

    raise FileNotFoundError(
        'Could not locate repo root containing surrogate/cost_model_collected.csv and surrogate/cost_model_run_history.csv. '
        f'started from: {start}'
    )


ROOT = find_repo_root(Path.cwd())
COLLECTED_CSV = ROOT / 'surrogate' / 'cost_model_collected.csv'
RUN_HISTORY_CSV = ROOT / 'surrogate' / 'cost_model_run_history.csv'

# ── Config ───────────────────────────────────────────────────────────────────
# Default to structured plan parsing; override via env var if desired.
PLAN_REPR            = os.environ.get('PLAN_REPR', 'structured')  # 'embedding' | 'structured'
REL_LEVELS           = 5      # relevance label granularity for LambdaRank
N_CV_FOLDS           = 5      # GroupKFold folds for PG baseline
PSEUDO_SPEARMAN_GATE = 0.0    # skip pseudo-labels for workloads with corr below this
PSEUDO_WEIGHT        = 0.3    # weight for pseudo-labelled MySQL rows

# ── Artifact versioning ──────────────────────────────────────────────────────
# Hash of key config so runs never silently overwrite each other.
_cfg_hash    = hashlib.md5(f'{PLAN_REPR}-{REL_LEVELS}-{RANDOM_SEED}'.encode()).hexdigest()[:8]
ARTIFACT_DIR = ROOT / 'surrogate' / 'artifacts' / 'transfer_rank_surrogate' / _cfg_hash
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('cwd         =', Path.cwd().resolve())
print('repo root   =', ROOT)
print('PLAN_REPR   =', PLAN_REPR)
print('cfg_hash    =', _cfg_hash)
print('Collected   =', COLLECTED_CSV)
print('Run history =', RUN_HISTORY_CSV)
print('Artifact dir=', ARTIFACT_DIR)


cwd         = /home/E2ETune-AI4DB/surrogate
repo root   = /home/E2ETune-AI4DB
PLAN_REPR   = structured
cfg_hash    = 6ecfae00
Collected   = /home/E2ETune-AI4DB/surrogate/cost_model_collected.csv
Run history = /home/E2ETune-AI4DB/surrogate/cost_model_run_history.csv
Artifact dir= /home/E2ETune-AI4DB/surrogate/artifacts/transfer_rank_surrogate/6ecfae00


In [2]:
# ── Parsing helpers ──────────────────────────────────────────────────────────

def safe_parse_json_list(value: Any) -> Optional[list]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, list):
        return value
    if not isinstance(value, str):
        return None
    s = value.strip()
    if not s:
        return None
    try:
        parsed = json.loads(s)
        return parsed if isinstance(parsed, list) else None
    except Exception:
        pass
    try:
        parsed = ast.literal_eval(s)
        return parsed if isinstance(parsed, list) else None
    except Exception:
        return None


def safe_parse_emb_vector(value: Any) -> Optional[np.ndarray]:
    lst = safe_parse_json_list(value)
    if lst is None:
        return None
    try:
        arr = np.asarray(lst, dtype=float)
        if arr.ndim != 1 or arr.size == 0:
            return None
        if not np.isfinite(arr).all():
            return None
        return arr
    except Exception:
        return None


# ── Load & merge data ────────────────────────────────────────────────────────
collected = pd.read_csv(COLLECTED_CSV, low_memory=False)
run_hist  = pd.read_csv(RUN_HISTORY_CSV, low_memory=False)

KEY_COL    = 'metadata.workload_key'
TARGET_COL = 'target.cost'

print('collected:', collected.shape)
print('run_hist :', run_hist.shape)

if KEY_COL not in run_hist.columns:
    raise KeyError(f"Missing {KEY_COL} in run history")
if KEY_COL not in collected.columns:
    raise KeyError(f"Missing {KEY_COL} in collected")

common_keys = [
    c for c in [
        'metadata.benchmark', 'metadata.db_engine', 'metadata.hardware',
        'metadata.hardware_specs.cores', 'metadata.hardware_specs.ram_gb',
        'metadata.hardware_specs.threads', 'metadata.workload', KEY_COL,
    ]
    if c in run_hist.columns and c in collected.columns
]

keep_cols = [
    c for c in collected.columns
    if c.startswith('collected.') or c.startswith('metadata.') or c == 'qp_emb_vector'
]

merged = run_hist.merge(collected[keep_cols], on=common_keys, how='left')

if TARGET_COL not in merged.columns:
    raise KeyError(f'Missing {TARGET_COL} after merge')

# ── Target preprocessing ─────────────────────────────────────────────────────
# `target.cost` mixes:
# - OLAP workloads: latency (positive)
# - OLTP workloads: negative throughput (negative)
# Convert to a single positive "latency-like" target where lower is better:
# - latency workloads: latency_like = cost (require cost > 0)
# - throughput workloads: latency_like = 1 / (-cost) (require cost < 0)

raw_cost = pd.to_numeric(merged[TARGET_COL], errors='coerce')
raw_cost = raw_cost.replace([np.inf, -np.inf], np.nan)

wk_series = merged[KEY_COL].astype(str)

wk_cost = pd.DataFrame({'wk': wk_series, 'c': raw_cost})
wk_stats = wk_cost.groupby('wk')['c'].agg(
    n='count',
    neg_frac=lambda s: float((s < 0).mean()),
    pos_frac=lambda s: float((s > 0).mean()),
    median='median',
)

wk_is_throughput = (wk_stats['median'] < 0)
is_throughput = wk_series.map(wk_is_throughput.to_dict()).fillna(False).to_numpy(dtype=bool)

raw_arr = raw_cost.to_numpy(dtype=float)
latency_like = np.full(len(raw_arr), np.nan, dtype=float)

lat_mask = ~is_throughput
lat_vals = raw_arr[lat_mask]
latency_like[lat_mask] = np.where(lat_vals > 0, lat_vals, np.nan)

thr_mask = is_throughput
thr = -raw_arr[thr_mask]
thr = np.where(thr > 0, thr, np.nan)
latency_like[thr_mask] = 1.0 / thr

latency_like = np.where(np.isfinite(latency_like), latency_like, np.nan)
keep_target = np.isfinite(latency_like)

dropped = int((~keep_target).sum())
if dropped:
    print(f'Dropping {dropped} rows with invalid {TARGET_COL} for latency-like target')

merged = merged.loc[keep_target].reset_index(drop=True)
y  = latency_like[keep_target].astype(float)
wk = merged[KEY_COL].astype(str).to_numpy()

print('objective type counts (workloads):', {
    'throughput_wk': int(wk_is_throughput.sum()),
    'latency_wk': int((~wk_is_throughput).sum()),
})
print('objective type counts (rows):', {
    'throughput_rows': int(thr_mask.sum()),
    'latency_rows': int(lat_mask.sum()),
})

print('merged:', merged.shape)
print('y(latency_like) stats - min:', float(np.min(y)), ' mean:', float(np.mean(y)), ' max:', float(np.max(y)))
print('unique workload keys:', len(np.unique(wk)))


collected: (264, 109)
run_hist : (22176, 81)
Dropping 104 rows with invalid target.cost for latency-like target
objective type counts (workloads): {'throughput_wk': 109, 'latency_wk': 154}
objective type counts (rows): {'throughput_rows': 7935, 'latency_rows': 14241}
merged: (22072, 182)
y(latency_like) stats - min: 2.1742046965080753e-05  mean: 2.1493904833769646  max: 112.7186926606944
unique workload keys: 263


In [3]:
# Phase 1.1 - Query plan encoding
#[NEW DEFAULT] If you already have embeddings in `qp_emb_vector`, you can skip this.
# Set PLAN_REPR=structured only if you want DB-agnostic structured plan features.

if PLAN_REPR != 'structured':
    # Keep placeholders so later cells can reference these names safely.
    plan_struct_cols   = []
    plan_struct_matrix = np.zeros((len(merged), 0), dtype=float)
    print('PLAN_REPR != structured: skipping structured plan parsing; using qp_emb_vector embeddings instead.')
else:
    # DB-agnostic structured features (optional)
    #
    # IMPROVEMENT: added `plan.parse_failed` flag so the model can learn when
    # plan signal is missing rather than silently receiving all-zeros.

    OP_ALIASES = {
        'seq scan': 'table_scan', 'table scan': 'table_scan',
        'index scan': 'index_scan', 'index only scan': 'index_scan',
        'hash join': 'join', 'merge join': 'join', 'nested loop': 'join', 'join': 'join',
        'sort': 'sort', 'aggregate': 'aggregate',
        'limit': 'limit', 'gather': 'gather', 'gather merge': 'gather',
    }


    def normalize_op(op: str) -> str:
        s = re.sub(r'\s+', ' ', (op or '').strip().lower())
        return OP_ALIASES.get(s, s)


    def structured_plan_features(plan_list: List[str]) -> Dict[str, float]:
        feats: Dict[str, float] = {}
        if not plan_list:
            feats['plan.parse_failed'] = 1.0
            return feats

        ops: List[str] = []
        costs: List[float] = []
        rows: List[float] = []
        any_index = join_hash = join_merge = join_nested = 0

        for p in plan_list:
            if not isinstance(p, str) or not p.strip():
                continue
            # Match operation names like "Hash Join(" or "Seq Scan("
            for raw in re.findall(r'([A-Za-z][A-Za-z ]+?)\(', p):
                norm = normalize_op(raw)
                ops.append(norm)
                rl = raw.lower()
                join_hash   += int('hash join'   in rl)
                join_merge  += int('merge join'  in rl)
                join_nested += int('nested loop' in rl)
                any_index    = any_index or int('index' in rl)
            for m in re.finditer(r'cost=([0-9]+(?:\.[0-9]+)?)', p):
                costs.append(float(m.group(1)))
            for m in re.finditer(r'rows=([0-9]+(?:\.[0-9]+)?)', p):
                rows.append(float(m.group(1)))

        if not ops:
            feats['plan.parse_failed'] = 1.0
            return feats

        total = float(len(ops))
        feats['plan.parse_failed']     = 0.0
        feats['plan.num_plans']        = float(len(plan_list))
        feats['plan.total_nodes']      = total
        feats['plan.any_index']        = float(any_index)
        feats['plan.join_hash_prop']   = float(join_hash)   / total
        feats['plan.join_merge_prop']  = float(join_merge)  / total
        feats['plan.join_nested_prop'] = float(join_nested) / total
        feats['plan.avg_cost']         = float(np.mean(costs)) if costs else 0.0
        feats['plan.max_cost']         = float(np.max(costs))  if costs else 0.0
        feats['plan.avg_rows']         = float(np.mean(rows))  if rows  else 0.0
        feats['plan.max_rows']         = float(np.max(rows))   if rows  else 0.0

        counts: Dict[str, int] = {}
        for op in ops:
            counts[op] = counts.get(op, 0) + 1
        for op, cnt in counts.items():
            feats[f'plan.op_prop.{op}'] = float(cnt) / total

        return feats


    plan_src = collected[[KEY_COL, 'collected.query_plans']].copy()
    plan_src['__plans'] = plan_src['collected.query_plans'].apply(safe_parse_json_list)

    feat_rows = []
    for wk_i, plans in zip(plan_src[KEY_COL].astype(str), plan_src['__plans']):
        f = structured_plan_features(plans or [])
        f[KEY_COL] = wk_i
        feat_rows.append(f)

    plan_struct_df = pd.DataFrame(feat_rows).fillna(0.0)
    merged_struct  = merged[[KEY_COL]].astype(str).merge(plan_struct_df, on=KEY_COL, how='left').fillna(0.0)

    plan_struct_cols   = [c for c in merged_struct.columns if c != KEY_COL]
    plan_struct_matrix = merged_struct[plan_struct_cols].to_numpy(dtype=float)

    parse_fail_rate = plan_struct_df.get('plan.parse_failed', pd.Series([0])).mean()
    print('structured plan dims:', plan_struct_matrix.shape)
    print('parse failure rate  :', f'{parse_fail_rate:.1%}')
    print('example plan cols   :', plan_struct_cols[:12])


structured plan dims: (22072, 63)
parse failure rate  : 14.0%
example plan cols   : ['plan.parse_failed', 'plan.num_plans', 'plan.total_nodes', 'plan.any_index', 'plan.join_hash_prop', 'plan.join_merge_prop', 'plan.join_nested_prop', 'plan.avg_cost', 'plan.max_cost', 'plan.avg_rows', 'plan.max_rows', 'plan.op_prop.aggregate']


In [4]:
# Phase 1.2 - Knob encoding  |  Phase 1.3 - Workload encoding
#
# IMPROVEMENT: drop near-constant columns before any model sees them.
# Constant features add noise and inflate the importance of real signals.

workload_cols_raw = [c for c in merged.columns if c.startswith('collected.workload_features.')]
knob_cols_raw     = [c for c in merged.columns if c.startswith('features.')]

LOG_HINTS         = ('buffer', 'mem', 'cache', 'size', 'capacity', 'tmp', 'wal', 'shared', 'work_mem')
log_knob_cols_raw = [c for c in knob_cols_raw if any(h in c.lower() for h in LOG_HINTS)]


def drop_near_constant(df: pd.DataFrame, cols: List[str], threshold: float = 0.99) -> List[str]:
    """Drop columns where >= threshold fraction of rows share the modal value."""
    keep = []
    for c in cols:
        arr = pd.to_numeric(df[c], errors='coerce').fillna(0)
        mode_frac = arr.value_counts(normalize=True).iloc[0] if len(arr) > 0 else 1.0
        if mode_frac < threshold:
            keep.append(c)
    dropped = set(cols) - set(keep)
    if dropped:
        print(f'  Dropped {len(dropped)} near-constant cols:', sorted(dropped)[:5],
              '...' if len(dropped) > 5 else '')
    return keep


print('Pruning near-constant knob columns...')
knob_cols     = drop_near_constant(merged, knob_cols_raw)
log_knob_cols = [c for c in log_knob_cols_raw if c in knob_cols]

print('Pruning near-constant workload columns...')
workload_cols = drop_near_constant(merged, workload_cols_raw)


def encode_knobs(
    df: pd.DataFrame,
    cols: List[str],
    log_cols: List[str],
) -> Tuple[np.ndarray, Dict[str, Dict[str, float]]]:
    Xk = df[cols].copy()
    for c in cols:
        Xk[c] = pd.to_numeric(Xk[c], errors='coerce')
    for c in log_cols:
        Xk[c] = np.log1p(Xk[c].where(Xk[c] >= 0))
    stats: Dict[str, Dict[str, float]] = {}
    for c in cols:
        arr   = Xk[c].to_numpy(dtype=float)
        mn    = float(np.nanmin(arr)) if np.isfinite(np.nanmin(arr)) else 0.0
        mx    = float(np.nanmax(arr)) if np.isfinite(np.nanmax(arr)) else mn
        denom = (mx - mn) if (mx - mn) != 0 else 1.0
        Xk[c] = ((Xk[c] - mn) / denom).clip(0.0, 1.0)
        stats[c] = {'min': mn, 'max': mx}
    return Xk.fillna(0.0).to_numpy(dtype=float), stats


knob_matrix,  knob_stats = encode_knobs(merged, knob_cols, log_knob_cols)
workload_matrix           = merged[workload_cols].fillna(0.0).to_numpy(dtype=float)

print('knob dims    :', knob_matrix.shape)
print('workload dims:', workload_matrix.shape)
print('log knobs    :', len(log_knob_cols))


Pruning near-constant knob columns...
Pruning near-constant workload columns...
  Dropped 5 near-constant cols: ['collected.workload_features.operator_proportions.max_agg', 'collected.workload_features.operator_proportions.min_agg', 'collected.workload_features.table_access_frequency.customer_summary', 'collected.workload_features.table_access_frequency.movie_info_idx', 'collected.workload_features.table_access_frequency.web_returns'] 
knob dims    : (22072, 71)
workload dims: (22072, 79)
log knobs    : 21


In [5]:
# Phase 1.4 - Final input: X = [plan_features, knobs, workload, hardware, db_type]
#
# IMPROVEMENT: `db_type` flag (0=PostgreSQL, 1=MySQL) is added HERE, BEFORE
# any model is trained. In v1 it was added only during MySQL transfer, causing
# a silent feature-dimension mismatch when warm-starting the ranker.

def get_plan_matrix(df: pd.DataFrame) -> Tuple[np.ndarray, Dict[str, object]]:
    if PLAN_REPR == 'structured':
        return plan_struct_matrix, {'plan_repr': 'structured', 'cols': plan_struct_cols}

    if 'qp_emb_vector' not in df.columns:
        raise KeyError(
            'Missing qp_emb_vector; run surrogate/add_query_plan_embeddings.py '
            'or set PLAN_REPR=structured'
        )
    vecs = df['qp_emb_vector'].apply(safe_parse_emb_vector).tolist()
    dims = [v.size for v in vecs if v is not None]
    if not dims:
        raise ValueError('No valid qp_emb_vector; set PLAN_REPR=structured or backfill.')

    dim = int(np.median(dims))
    mat = np.zeros((len(vecs), dim), dtype=float)
    missing = 0
    for i, v in enumerate(vecs):
        if v is None:
            missing += 1
            continue
        if v.size != dim:
            raise ValueError(f'Embedding dim mismatch at row {i}: {v.size} != {dim}')
        mat[i, :] = v

    return mat, {'plan_repr': 'embedding', 'dim': dim, 'missing': missing}


plan_matrix, plan_meta = get_plan_matrix(merged)

engine = (
    merged['metadata.db_engine'].astype(str).str.lower()
    if 'metadata.db_engine' in merged.columns
    else pd.Series(['postgresql'] * len(merged))
)
db_type_flag = (engine == 'mysql').astype(int).to_numpy().reshape(-1, 1)  # 0=PG, 1=MySQL

# Hardware specs (RAM/cores/threads) to support cross-machine generalization.
_hw_cols_candidates = [
    'metadata.hardware_specs.ram_gb',
    'metadata.hardware_specs.cores',
    'metadata.hardware_specs.threads',
]
hardware_cols = [c for c in _hw_cols_candidates if c in merged.columns]


def encode_hardware_specs(df: pd.DataFrame, cols: List[str]) -> np.ndarray:
    if not cols:
        return np.zeros((len(df), 0), dtype=float)
    Xh = df[cols].copy()
    for c in cols:
        Xh[c] = pd.to_numeric(Xh[c], errors='coerce')
        # log1p compresses scale differences (esp. RAM) while keeping monotonicity.
        Xh[c] = np.log1p(Xh[c].where(Xh[c] >= 0))
        arr = Xh[c].to_numpy(dtype=float)
        mn = float(np.nanmin(arr)) if np.isfinite(np.nanmin(arr)) else 0.0
        mx = float(np.nanmax(arr)) if np.isfinite(np.nanmax(arr)) else mn
        denom = (mx - mn) if (mx - mn) != 0 else 1.0
        Xh[c] = ((Xh[c] - mn) / denom).clip(0.0, 1.0)
    return Xh.fillna(0.0).to_numpy(dtype=float)


hardware_matrix = encode_hardware_specs(merged, hardware_cols)

X = np.hstack([plan_matrix, knob_matrix, workload_matrix, hardware_matrix, db_type_flag]).astype(float)

if plan_meta.get('plan_repr') == 'structured':
    plan_feature_names = list(plan_meta.get('cols', []))
else:
    dim = int(plan_meta['dim'])
    plan_feature_names = [f'plan.emb.{i}' for i in range(dim)]

FEATURE_NAMES = plan_feature_names + knob_cols + workload_cols + hardware_cols + ['db_type']

print('plan dims    :', plan_matrix.shape)
print('knob dims    :', knob_matrix.shape)
print('workload dims:', workload_matrix.shape)
print('hw dims      :', hardware_matrix.shape, '(cols:', hardware_cols, ')')
print('X dims       :', X.shape)
print('plan_meta    :', plan_meta)
print('PG rows      :', int((engine == 'postgresql').sum()))
print('MySQL rows   :', int((engine == 'mysql').sum()))


plan dims    : (22072, 63)
knob dims    : (22072, 71)
workload dims: (22072, 79)
hw dims      : (22072, 3) (cols: ['metadata.hardware_specs.ram_gb', 'metadata.hardware_specs.cores', 'metadata.hardware_specs.threads'] )
X dims       : (22072, 217)
plan_meta    : {'plan_repr': 'structured', 'cols': ['plan.parse_failed', 'plan.num_plans', 'plan.total_nodes', 'plan.any_index', 'plan.join_hash_prop', 'plan.join_merge_prop', 'plan.join_nested_prop', 'plan.avg_cost', 'plan.max_cost', 'plan.avg_rows', 'plan.max_rows', 'plan.op_prop.aggregate', 'plan.op_prop.gather', 'plan.op_prop.table_scan', 'plan.op_prop.sort', 'plan.op_prop.join', 'plan.op_prop.hash', 'plan.op_prop.limit', 'plan.op_prop.index_scan', 'plan.op_prop.bitmap heap scan', 'plan.op_prop.bitmap index scan', 'plan.op_prop.materialize', 'plan.op_prop.cte scan', 'plan.op_prop.series', 'plan.op_prop.lower', 'plan.op_prop.upper', 'plan.op_prop.as g', 'plan.op_prop.any', 'plan.op_prop.subquery scan', 'plan.op_prop.unique', 'plan.op_prop.a

In [6]:
# Shared helpers

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

# MAPE can be wildly misleading when y_true is near zero.
# We use a small *data-driven* denominator floor (quantile of |y_true|) to stabilize it.
MAPE_DENOM_FLOOR_Q = 0.999  # 25th percentile of |y_true| within the evaluated set


def spearman_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    s1 = pd.Series(y_true).rank(method='average')
    s2 = pd.Series(y_pred).rank(method='average')
    return float(s1.corr(s2))


def mape_pct(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    eps: float = 1e-9,
    denom_floor_q: float = MAPE_DENOM_FLOOR_Q,
) -> float:
    """Mean Absolute Percentage Error in percent (stabilized).

    Vanilla MAPE uses |y_true| in the denominator and can explode when y_true≈0.
    This version floors the denominator at a small quantile of |y_true| (per-call)
    to keep the metric interpretable for mixed/normalized targets.
    """
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)

    abs_y = np.abs(yt)
    finite = np.isfinite(abs_y)
    abs_y_f = abs_y[finite]

    if abs_y_f.size == 0:
        floor = 0.0
    else:
        # quantile may be 0 if many near-zero values; eps still protects division.
        floor = float(np.quantile(abs_y_f, float(denom_floor_q)))

    denom = np.maximum(abs_y, max(float(eps), floor))
    return float(np.mean(np.abs(yt - yp) / denom) * 100.0)


def spearman_mean_per_workload(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    wk_keys: np.ndarray,
) -> Tuple[float, int, int]:
    """Mean Spearman across workloads; returns (mean, n_valid, n_total).
    Spearman is only defined when a workload has >=2 samples and non-constant preds/targets."""
    df = pd.DataFrame({'wk': wk_keys, 'y': y_true, 'p': y_pred})
    vals = []
    n_total = 0
    for _, g in df.groupby('wk'):
        if len(g) < 2:
            continue
        n_total += 1
        s = spearman_corr(g['y'].to_numpy(), g['p'].to_numpy())
        if np.isfinite(s):
            vals.append(float(s))
    mean = float(np.mean(vals)) if vals else float('nan')
    return mean, int(len(vals)), int(n_total)


def regression_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    wk_keys: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    rmse = float(math.sqrt(mean_squared_error(y_true, y_pred)))
    mape = float(mape_pct(y_true, y_pred))
    sp_pool = float(spearman_corr(y_true, y_pred))
    out: Dict[str, float] = {
        'rmse': rmse,
        'mape_pct': mape,
        'spearman_pool': sp_pool,
    }
    if wk_keys is not None:
        sp_wk, n_valid, n_total = spearman_mean_per_workload(y_true, y_pred, wk_keys)
        out['spearman_wk_mean'] = float(sp_wk)
        out['spearman_wk_valid'] = float(n_valid)
        out['spearman_wk_total'] = float(n_total)
    return out


def eval_per_workload(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    wk_keys: np.ndarray,
) -> Dict[str, float]:
    """Ranking-style per-workload metrics used in Phase 4 (kept for rankers)."""
    df = pd.DataFrame({'wk': wk_keys, 'y': y_true, 'p': y_pred})
    spears, top1, top3 = [], [], []
    n_workloads = 0
    for _, g in df.groupby('wk'):
        if len(g) < 2:
            continue
        n_workloads += 1
        s = spearman_corr(g['y'].to_numpy(), g['p'].to_numpy())
        spears.append(s)
        best_true = int(g['y'].idxmin())
        top1_pred = int(g['p'].idxmin())
        top3_pred = g.nsmallest(min(3, len(g)), 'p').index.tolist()
        top1.append(1.0 if top1_pred == best_true else 0.0)
        top3.append(1.0 if best_true in top3_pred else 0.0)

    spears_valid = [s for s in spears if np.isfinite(s)]
    spearman_mean = float(np.mean(spears_valid)) if spears_valid else float('nan')
    return {
        'spearman_mean'   : spearman_mean,
        'top1_acc'        : float(np.mean(top1)) if top1 else float('nan'),
        'top3_recall'     : float(np.mean(top3)) if top3 else float('nan'),
        'n_workloads'     : int(n_workloads),
        'n_spearman_valid': int(len(spears_valid)),
    }


def group_sizes(sorted_wk: np.ndarray) -> List[int]:
    sizes, last, cnt = [], None, 0
    for k in sorted_wk:
        if last is None:
            last, cnt = k, 1
        elif k == last:
            cnt += 1
        else:
            sizes.append(cnt)
            last, cnt = k, 1
    if last is not None:
        sizes.append(cnt)
    return sizes


def relevance_labels(
    lat: np.ndarray,
    wk_keys: np.ndarray,
    levels: int = REL_LEVELS,
) -> np.ndarray:
    """Lower latency -> higher relevance label (levels-1 = best)."""
    rel = np.zeros_like(lat, dtype=int)
    df  = pd.DataFrame({'wk': wk_keys, 'y': lat})
    for _, idx in df.groupby('wk').groups.items():
        y_g   = df.loc[idx, 'y']
        ranks = (y_g.rank(method='first', ascending=True).to_numpy() - 1)
        if len(ranks) == 1:
            rel[idx] = levels - 1
            continue
        q        = ranks / max(1, (len(ranks) - 1))
        rel[idx] = np.floor((1.0 - q) * (levels - 1) + 1e-9).astype(int)
    return rel


print('Helpers loaded.')


Helpers loaded.


In [7]:
# Phase 2 - Regression baseline on PostgreSQL
#
# IMPROVEMENTS vs v1:
#   1. Log1p-transform target - right-skewed latency distributions hurt RMSE.
#      Invert with expm1 at eval time so RMSE stays interpretable.
#   2. Full GroupKFold CV (all N_CV_FOLDS folds), not just next(iter(...)).
#   3. Early stopping on an inner val split within each fold.
#   4. Feature importance diagnostics: checks plan vs knob vs workload split.
#   5. Final regressor retrained on ALL PG data (used for pseudo-label quality later).

try:
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
    _HAS_LGBM = True
except Exception:
    from sklearn.ensemble import HistGradientBoostingRegressor
    _HAS_LGBM = False
    print('WARNING: LightGBM not found - falling back to HistGradientBoostingRegressor (no early stopping).')

is_pg  = (engine == 'postgresql').to_numpy()
X_pg   = X[is_pg]
y_pg   = y[is_pg]
wk_pg  = wk[is_pg]

# IMPROVEMENT 1: log-transform the target
y_pg_log = np.log1p(y_pg)

n_splits = min(N_CV_FOLDS, len(np.unique(wk_pg)))
gkf      = GroupKFold(n_splits=n_splits)

cv_rmse, cv_mape, cv_spearman, fi_accum = [], [], [], None

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_pg, y_pg_log, groups=wk_pg)):
    X_tr, X_te = X_pg[tr_idx], X_pg[te_idx]
    y_tr, y_te = y_pg_log[tr_idx], y_pg_log[te_idx]
    wk_te      = wk_pg[te_idx]

    # Inner val split for early stopping (last 15% of training workload groups)
    tr_groups    = np.unique(wk_pg[tr_idx])
    n_val_groups = max(1, int(0.15 * len(tr_groups)))
    val_groups   = set(tr_groups[-n_val_groups:])
    val_mask     = np.array([g in val_groups for g in wk_pg[tr_idx]])

    X_tr2, X_val = X_tr[~val_mask], X_tr[val_mask]
    y_tr2, y_val = y_tr[~val_mask], y_tr[val_mask]

    if _HAS_LGBM:
        reg = LGBMRegressor(
            n_estimators     = 3000,
            learning_rate    = 0.03,
            num_leaves       = 63,
            min_child_samples= 20,
            lambda_l1        = 0.1,
            lambda_l2        = 0.1,
            subsample        = 0.9,
            colsample_bytree = 0.9,
            random_state     = RANDOM_SEED,
            n_jobs           = -1,
        )
        reg.fit(
            X_tr2, y_tr2,
            eval_set=[(X_val, y_val)],
            callbacks=[early_stopping(50, verbose=False), log_evaluation(0)],
        )
    else:
        reg = HistGradientBoostingRegressor(random_state=RANDOM_SEED)
        reg.fit(X_tr2, y_tr2)

    pred_log = reg.predict(X_te)
    pred     = np.expm1(pred_log)   # invert log1p for interpretable metrics
    y_te_raw = np.expm1(y_te)

    rmse_fold = math.sqrt(mean_squared_error(y_te_raw, pred))
    mape_fold = mape_pct(y_te_raw, pred)
    sp_fold   = spearman_corr(y_te_raw, pred)

    cv_rmse.append(rmse_fold)
    cv_mape.append(mape_fold)
    cv_spearman.append(sp_fold)

    if _HAS_LGBM and hasattr(reg, 'feature_importances_'):
        fi = reg.feature_importances_
        fi_accum = fi if fi_accum is None else fi_accum + fi

    best_it = getattr(reg, 'best_iteration_', '-') if _HAS_LGBM else '-'
    print(
        f'  Fold {fold+1}/{n_splits}  '
        f'RMSE: {rmse_fold:.4f}  '
        f'MAPE: {mape_fold:.2f}%  '
        f'Spearman: {sp_fold:.4f}  '
        f'best_iter: {best_it}'
    )

print(f'\nCV RMSE     : {np.mean(cv_rmse):.4f} +/- {np.std(cv_rmse):.4f}')
print(f'CV MAPE (%) : {np.mean(cv_mape):.2f} +/- {np.std(cv_mape):.2f}')
print(f'CV Spearman : {np.mean(cv_spearman):.4f} +/- {np.std(cv_spearman):.4f}')

# IMPROVEMENT 4: Feature importance diagnostics
if fi_accum is not None and len(FEATURE_NAMES) == len(fi_accum):
    fi_df    = pd.DataFrame({'feature': FEATURE_NAMES, 'importance': fi_accum})
    fi_df    = fi_df.sort_values('importance', ascending=False)
    total_fi = fi_df['importance'].sum() or 1.0

    plan_fi = fi_df[fi_df['feature'].str.startswith('plan.')]['importance'].sum()
    knob_fi = fi_df[fi_df['feature'].str.startswith('features.')]['importance'].sum()
    wkld_fi = fi_df[fi_df['feature'].str.startswith('collected.workload')]['importance'].sum()
    hw_fi   = fi_df[fi_df['feature'].str.startswith('metadata.hardware_specs.')]['importance'].sum()
    db_fi   = fi_df[fi_df['feature'] == 'db_type']['importance'].sum()

    print(f'\nFeature group importance (summed across folds):')
    print(f'  Plan features     : {plan_fi/total_fi:.1%}')
    print(f'  Knob features     : {knob_fi/total_fi:.1%}')
    print(f'  Workload features : {wkld_fi/total_fi:.1%}')
    print(f'  Hardware specs    : {hw_fi/total_fi:.1%}')
    print(f'  db_type flag      : {db_fi/total_fi:.1%}')
    print(f'\nTop-15 features:')
    print(fi_df.head(15).to_string(index=False))

    if plan_fi / total_fi < 0.05:
        print('\nWARNING: Plan features < 5% of total importance.')
        print('  Check plan parsing or try PLAN_REPR=embedding.')

# IMPROVEMENT 5: Retrain final PG regressor on ALL PG data
if _HAS_LGBM:
    best_n    = getattr(reg, 'best_iteration_', 1000)
    reg_final = LGBMRegressor(
        n_estimators=best_n, learning_rate=0.03, num_leaves=63,
        min_child_samples=20, lambda_l1=0.1, lambda_l2=0.1,
        subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_SEED, n_jobs=-1,
    )
else:
    reg_final = HistGradientBoostingRegressor(random_state=RANDOM_SEED)

reg_final.fit(X_pg, y_pg_log)
print('\nFinal PG regressor trained on all PG data.')


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007538 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11832
[LightGBM] [Info] Number of data points in the train set: 14637, number of used features: 153
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Start training from score 0.602691


/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
  Fold 1/5  RMSE: 0.8445  MAPE: 6.97%  Spearman: 0.9366  best_iter: 445
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004134 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11841
[LightGBM] [Info] Number of data points in the train set: 14452, number of used features: 152
[LightGBM] [Warn

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


...

[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
  Fold 3/5  RMSE: 1.0307  MAPE: 4.10%  Spearman: 0.9549  best_iter: 254
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004448 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11811
[LightGBM] [Info] Number of data points in the train set: 14468, number of used features: 152
[LightGBM] [Warn

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
  Fold 4/5  RMSE: 1.3469  MAPE: 7.00%  Spearman: 0.9285  best_iter: 261
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005819 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11867
[LightGBM] [Info] Number of data points in the train set: 14414, number of used features: 153
[LightGBM] [Warn

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
  Fold 5/5  RMSE: 0.7979  MAPE: 4.65%  Spearman: 0.9642  best_iter: 262

CV RMSE     : 0.9710 +/- 0.2046
CV MAPE (%) : 5.54 +/- 1.21
CV Spearman : 0.9417 +/- 0.0154

Feature group importance (summed across folds):
  Plan features     : 42.8%
  Knob features     : 45.1%
  Workload features : 11.3%
  Hardware specs    : 0.8%
  db_type flag      : 0.0%

Top-15 features:
                                                   feature  importance
                                   features.shared_buffers       10803
                                             plan.avg_cost        6809
                                             plan.max_cost        6610
                                         features.work_mem        5153
                                            plan.num_plans        48

In [8]:
# Phase 2b - Joint regression (PG+MySQL) with MySQL upweight (unseen-workload MySQL split)
#
# Metrics: RMSE + MAPE + Spearman (pooled + mean-per-workload).

# --- Build a held-out MySQL workload split (by workload group) ---
is_my = (engine == 'mysql').to_numpy()
is_pg = (engine == 'postgresql').to_numpy()

if not np.any(is_my):
    print('No MySQL rows found; skipping Phase 2b (joint regression).')
else:
    X_my_all = X[is_my]
    y_my_all = y[is_my]
    wk_my_all = wk[is_my]

    unique_my_wks = np.unique(wk_my_all)
    n_test_wks = max(1, int(0.2 * len(unique_my_wks)))
    rng = np.random.default_rng(RANDOM_SEED)
    test_wks_joint = set(rng.choice(unique_my_wks, size=n_test_wks, replace=False))
    ft_mask = np.array([w not in test_wks_joint for w in wk_my_all])
    te_mask = ~ft_mask

    X_my_ft, y_my_ft, wk_my_ft = X_my_all[ft_mask], y_my_all[ft_mask], wk_my_all[ft_mask]
    X_my_te, y_my_te, wk_my_te = X_my_all[te_mask], y_my_all[te_mask], wk_my_all[te_mask]

    X_pg_all = X[is_pg]
    y_pg_all = y[is_pg]
    wk_pg_all = wk[is_pg]

    print('Joint split (MySQL workloads):')
    print(f'  MySQL fine-tune: {len(X_my_ft)} rows / {len(np.unique(wk_my_ft))} workloads')
    print(f'  MySQL held-out : {len(X_my_te)} rows / {len(np.unique(wk_my_te))} workloads')
    print(f'  PG rows        : {len(X_pg_all)}')

    # --- Train joint model ---
    # Upweight MySQL so it isn't drowned out by PG rows.
    mysql_weight = float(len(X_pg_all) / max(1, len(X_my_ft)))
    mysql_weight = max(1.0, mysql_weight)
    w_pg = np.ones(len(X_pg_all), dtype=float)
    w_my = np.ones(len(X_my_ft), dtype=float) * mysql_weight

    X_train = np.vstack([X_pg_all, X_my_ft])
    y_train = np.concatenate([y_pg_all, y_my_ft])
    wk_train = np.concatenate([wk_pg_all, wk_my_ft])
    w_train = np.concatenate([w_pg, w_my])

    y_train_log = np.log1p(y_train)

    # Group-aware validation split for early stopping
    uniq_groups = np.unique(wk_train)
    rng.shuffle(uniq_groups)
    n_val_groups = max(1, int(0.15 * len(uniq_groups)))
    val_groups = set(uniq_groups[:n_val_groups])
    val_mask = np.array([g in val_groups for g in wk_train])

    X_tr2, X_val = X_train[~val_mask], X_train[val_mask]
    y_tr2, y_val = y_train_log[~val_mask], y_train_log[val_mask]
    w_tr2, w_val = w_train[~val_mask], w_train[val_mask]

    if not _HAS_LGBM:
        raise RuntimeError('Phase 2b requires LightGBM (pip install lightgbm).')

    reg_joint = LGBMRegressor(
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=20,
        lambda_l1=0.1,
        lambda_l2=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    reg_joint.fit(
        X_tr2, y_tr2,
        sample_weight=w_tr2,
        eval_set=[(X_val, y_val)],
        eval_sample_weight=[w_val],
        callbacks=[early_stopping(100, verbose=False), log_evaluation(0)],
    )

    best_it = getattr(reg_joint, 'best_iteration_', None)
    print(f'\nJoint regressor trained. mysql_weight={mysql_weight:.2f}  best_iter={best_it}')

    def eval_regressor_on_mysql(name: str, reg_model) -> Dict[str, float]:
        pred_log = reg_model.predict(X_my_te)
        pred = np.expm1(pred_log)
        m = regression_metrics(y_my_te, pred, wk_keys=wk_my_te)
        wk_valid = int(m.get('spearman_wk_valid', 0.0))
        wk_total = int(m.get('spearman_wk_total', 0.0))
        print(f"\n[{name}] Held-out MySQL (unseen workloads):")
        print(f"  RMSE                 : {m['rmse']:.4f}")
        print(f"  MAPE (%)             : {m['mape_pct']:.2f}")
        print(f"  Spearman (pooled)    : {m['spearman_pool']:.4f}")
        print(f"  Spearman (wk mean)   : {m['spearman_wk_mean']:.4f}  ({wk_valid}/{wk_total} valid)")
        return m

    if 'reg_final' in globals():
        _ = eval_regressor_on_mysql('PG-only reg_final', reg_final)
    else:
        print('WARNING: reg_final not found; skipping PG-only baseline.')

    _ = eval_regressor_on_mysql('Joint weighted regressor', reg_joint)


Joint split (MySQL workloads):
  MySQL fine-tune: 1091 rows / 37 workloads
  MySQL held-out : 223 rows / 9 workloads
  PG rows        : 20758
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006598 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13249
[LightGBM] [Info] Number of data points in the train set: 18683, number of used features: 206
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Start training from score 1.324938

Joint regressor trained. mysql_weight=19.03  best_iter=868
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Curr

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2855: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2855: RuntimeWarning: inval

In [9]:
# Phase 2c - Within-workload MySQL evaluation (BO-like) + stronger transfer variants
#
# Goal: measure how well the surrogate ranks configs *within the same workload*,
# which better matches Bayesian optimization (fixed workload, many configs).
#
# IMPORTANT: we split *configs within each workload* (train/test) so the model must
# learn knob effects to generalize to new configs for the same workload.
#
# Metrics: RMSE + MAPE + Spearman (pooled + mean-per-workload).

def within_workload_random_split(
    wk_keys: np.ndarray,
    test_frac: float = 0.2,
    seed: int = RANDOM_SEED,
    min_train: int = 2,
    min_test: int = 2,
) -> Tuple[np.ndarray, np.ndarray]:
    rng_local = np.random.default_rng(seed)
    train_idx, test_idx = [], []

    groups = pd.Series(np.arange(len(wk_keys))).groupby(wk_keys).groups
    for wk_key, g in groups.items():
        idx = np.array(list(g), dtype=int)
        n = len(idx)
        if n < 2:
            train_idx.extend(idx.tolist())
            continue

        n_test = int(round(test_frac * n))
        if n >= (min_train + min_test):
            n_test = max(min_test, n_test)
            n_test = min(n - min_train, n_test)
        else:
            n_test = min(max(1, n_test), n - 1)

        chosen = rng_local.choice(idx, size=n_test, replace=False)
        chosen_set = set(int(x) for x in chosen.tolist())
        for i in idx.tolist():
            if int(i) in chosen_set:
                test_idx.append(int(i))
            else:
                train_idx.append(int(i))

    return np.array(train_idx, dtype=int), np.array(test_idx, dtype=int)


def assign_within_workload_folds(
    wk_keys: np.ndarray,
    n_splits: int = 5,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """Assign each sample a fold id, stratified within each workload group.

    This creates BO-like CV: each workload contributes test points in each fold,
    so we evaluate generalization to new configs within the same workload.
    """
    rng_local = np.random.default_rng(seed)
    fold_id = np.empty(len(wk_keys), dtype=int)
    groups = pd.Series(np.arange(len(wk_keys))).groupby(wk_keys).groups
    for _, g in groups.items():
        idx = np.array(list(g), dtype=int)
        rng_local.shuffle(idx)
        fold_id[idx] = (np.arange(len(idx)) % n_splits)
    return fold_id


if not np.any(is_my):
    print('No MySQL rows found; skipping Phase 2c.')
else:
    X_my_all = X[is_my]
    y_my_all = y[is_my]
    wk_my_all = wk[is_my]

    TEST_FRAC = 0.2
    tr_idx, te_idx = within_workload_random_split(wk_my_all, test_frac=TEST_FRAC, seed=RANDOM_SEED)

    X_my_tr, y_my_tr, wk_my_tr = X_my_all[tr_idx], y_my_all[tr_idx], wk_my_all[tr_idx]
    X_my_te, y_my_te, wk_my_te = X_my_all[te_idx], y_my_all[te_idx], wk_my_all[te_idx]

    print('Within-workload MySQL split:')
    print(f'  test_frac          : {TEST_FRAC}')
    print(f'  train rows         : {len(X_my_tr)}')
    print(f'  test rows          : {len(X_my_te)}')
    print(f'  workloads (train)  : {len(np.unique(wk_my_tr))}')
    print(f'  workloads (test)   : {len(np.unique(wk_my_te))}')

    if not _HAS_LGBM:
        raise RuntimeError('Phase 2c requires LightGBM (pip install lightgbm).')

    def fit_lgbm_reg(
        X_tr: np.ndarray,
        y_tr_log: np.ndarray,
        sample_weight: Optional[np.ndarray] = None,
        objective: str = 'regression',
        extra_params: Optional[Dict[str, object]] = None,
    ) -> LGBMRegressor:
        rng_local = np.random.default_rng(RANDOM_SEED)
        idx = np.arange(len(X_tr))
        rng_local.shuffle(idx)
        n_val = max(1, int(0.15 * len(idx)))
        val_i = idx[:n_val]
        tr_i = idx[n_val:]

        params = dict(
            objective=objective,
            n_estimators=5000,
            learning_rate=0.03,
            num_leaves=63,
            min_child_samples=20,
            lambda_l1=0.1,
            lambda_l2=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
        if extra_params:
            params.update(extra_params)

        reg = LGBMRegressor(**params)
        reg.fit(
            X_tr[tr_i], y_tr_log[tr_i],
            sample_weight=None if sample_weight is None else sample_weight[tr_i],
            eval_set=[(X_tr[val_i], y_tr_log[val_i])],
            eval_sample_weight=None if sample_weight is None else [sample_weight[val_i]],
            callbacks=[early_stopping(100, verbose=False), log_evaluation(0)],
        )
        return reg

    def eval_model(name: str, pred_log: np.ndarray) -> Dict[str, float]:
        pred = np.expm1(pred_log)
        m = regression_metrics(y_my_te, pred, wk_keys=wk_my_te)
        wk_valid = int(m.get('spearman_wk_valid', 0.0))
        wk_total = int(m.get('spearman_wk_total', 0.0))
        print(f"\n[{name}] Within-workload MySQL:")
        print(f"  RMSE                 : {m['rmse']:.4f}")
        print(f"  MAPE (%)             : {m['mape_pct']:.2f}")
        print(f"  Spearman (pooled)    : {m['spearman_pool']:.4f}")
        print(f"  Spearman (wk mean)   : {m['spearman_wk_mean']:.4f}  ({wk_valid}/{wk_total} valid)")
        return m

    def summarize_cv(name: str, fold_metrics: List[Dict[str, float]]) -> None:
        def _mean_std(key: str) -> Tuple[float, float]:
            vals = np.array([m.get(key, float('nan')) for m in fold_metrics], dtype=float)
            vals = vals[np.isfinite(vals)]
            if len(vals) == 0:
                return float('nan'), float('nan')
            return float(np.mean(vals)), float(np.std(vals))

        rmse_mu, rmse_sd = _mean_std('rmse')
        mape_mu, mape_sd = _mean_std('mape_pct')
        sp_mu, sp_sd = _mean_std('spearman_pool')
        wk_mu, wk_sd = _mean_std('spearman_wk_mean')

        wk_valid = np.array([m.get('spearman_wk_valid', float('nan')) for m in fold_metrics], dtype=float)
        wk_total = np.array([m.get('spearman_wk_total', float('nan')) for m in fold_metrics], dtype=float)
        wk_valid_sum = int(np.nansum(wk_valid))
        wk_total_sum = int(np.nansum(wk_total))

        print(f"\n[{name}] 5-fold CV (within-workload; fold-mean ± std):")
        print(f"  RMSE                 : {rmse_mu:.4f} ± {rmse_sd:.4f}")
        print(f"  MAPE (%)             : {mape_mu:.2f} ± {mape_sd:.2f}")
        print(f"  Spearman (pooled)    : {sp_mu:.4f} ± {sp_sd:.4f}")
        print(f"  Spearman (wk mean)   : {wk_mu:.4f} ± {wk_sd:.4f}  ({wk_valid_sum}/{wk_total_sum} valid across folds)")

    y_my_tr_log = np.log1p(y_my_tr)

    # 1) PG-only baseline (zero-shot)
    if 'reg_final' in globals():
        pred_pg_log = reg_final.predict(X_my_te)
        _ = eval_model('PG-only reg_final (zero-shot)', pred_pg_log)
    else:
        pred_pg_log = None
        print('WARNING: reg_final not found; skipping PG-only baseline.')

    # 2) MySQL-only regressor
    reg_my = fit_lgbm_reg(X_my_tr, y_my_tr_log)
    pred_my_log = reg_my.predict(X_my_te)
    _ = eval_model('MySQL-only regressor', pred_my_log)

    # 3) Joint weighted regressor (PG + MySQL, with MySQL upweighted)
    if np.any(is_pg):
        mysql_weight = float(len(X_pg_all) / max(1, len(X_my_tr)))
        mysql_weight = max(1.0, mysql_weight)
        X_joint = np.vstack([X_pg_all, X_my_tr])
        y_joint_log = np.log1p(np.concatenate([y_pg_all, y_my_tr]))
        w_joint = np.concatenate([np.ones(len(X_pg_all)), np.ones(len(X_my_tr)) * mysql_weight])
        reg_joint_ww = fit_lgbm_reg(X_joint, y_joint_log, sample_weight=w_joint)
        pred_joint_ww_log = reg_joint_ww.predict(X_my_te)
        _ = eval_model(f'Joint weighted regressor (mysql_weight={mysql_weight:.2f})', pred_joint_ww_log)

    # 4) Residual adapter: final_pred = PG_pred + residual_model(X)
    if pred_pg_log is not None:
        pred_pg_tr_log = reg_final.predict(X_my_tr)
        resid_tr = y_my_tr_log - pred_pg_tr_log
        reg_resid = fit_lgbm_reg(X_my_tr, resid_tr)
        resid_te = reg_resid.predict(X_my_te)
        pred_adapter_log = pred_pg_log + resid_te
        _ = eval_model('PG→MySQL residual adapter', pred_adapter_log)

    # 5) Workload-centered training
    mu_by_wk = pd.Series(y_my_tr_log).groupby(pd.Series(wk_my_tr)).mean().to_dict()
    fallback_mu = float(np.mean(y_my_tr_log))
    mu_tr = np.array([mu_by_wk.get(k, fallback_mu) for k in wk_my_tr], dtype=float)
    mu_te = np.array([mu_by_wk.get(k, fallback_mu) for k in wk_my_te], dtype=float)
    y_center_tr = y_my_tr_log - mu_tr

    reg_center = fit_lgbm_reg(X_my_tr, y_center_tr)
    pred_center_te = reg_center.predict(X_my_te)
    pred_center_log = pred_center_te + mu_te
    _ = eval_model('MySQL workload-centered regressor', pred_center_log)

    # 6) Workload-standardized training (center + divide by per-workload std)
    sigma_by_wk = pd.Series(y_my_tr_log).groupby(pd.Series(wk_my_tr)).std(ddof=0).to_dict()
    fallback_sigma = float(np.std(y_my_tr_log, ddof=0)) or 1.0
    sigma_tr = np.array([max(float(sigma_by_wk.get(k, fallback_sigma)), 1e-6) for k in wk_my_tr], dtype=float)
    sigma_te = np.array([max(float(sigma_by_wk.get(k, fallback_sigma)), 1e-6) for k in wk_my_te], dtype=float)
    y_std_tr = (y_my_tr_log - mu_tr) / sigma_tr

    reg_std = fit_lgbm_reg(X_my_tr, y_std_tr)
    pred_std_te = reg_std.predict(X_my_te)
    pred_std_log = pred_std_te * sigma_te + mu_te
    _ = eval_model('MySQL workload-standardized regressor', pred_std_log)

    # --- 5-fold within-workload cross-validation ---
    # NOTE: For workload-centered/standardized methods, evaluating Spearman over *global* out-of-fold
    # predictions can be misleading, because each fold reconstructs outputs using a different per-workload
    # baseline (mu/sigma). That can scramble within-workload ranks across folds.
    # We therefore report fold-wise metrics and summarize mean ± std.

    N_SPLITS = 5
    fold_id = assign_within_workload_folds(wk_my_all, n_splits=N_SPLITS, seed=RANDOM_SEED)
    y_my_all_log = np.log1p(y_my_all)

    # CV 2) MySQL-only regressor
    fold_ms = []
    for f in range(N_SPLITS):
        tr = fold_id != f
        te = fold_id == f
        reg = fit_lgbm_reg(X_my_all[tr], y_my_all_log[tr])
        pred_log = reg.predict(X_my_all[te])
        pred = np.expm1(pred_log)
        fold_ms.append(regression_metrics(y_my_all[te], pred, wk_keys=wk_my_all[te]))
    summarize_cv('MySQL-only regressor', fold_ms)

    # CV 5) Workload-centered regressor
    fold_ms = []
    for f in range(N_SPLITS):
        tr = fold_id != f
        te = fold_id == f

        mu_by_wk = pd.Series(y_my_all_log[tr]).groupby(pd.Series(wk_my_all[tr])).mean().to_dict()
        fallback_mu = float(np.mean(y_my_all_log[tr]))
        mu_tr = np.array([mu_by_wk.get(k, fallback_mu) for k in wk_my_all[tr]], dtype=float)
        mu_te = np.array([mu_by_wk.get(k, fallback_mu) for k in wk_my_all[te]], dtype=float)

        y_center_tr = y_my_all_log[tr] - mu_tr
        reg = fit_lgbm_reg(X_my_all[tr], y_center_tr)
        pred_center_te = reg.predict(X_my_all[te])
        pred_log = pred_center_te + mu_te
        pred = np.expm1(pred_log)
        fold_ms.append(regression_metrics(y_my_all[te], pred, wk_keys=wk_my_all[te]))
    summarize_cv('MySQL workload-centered regressor', fold_ms)

    # CV 6) Workload-standardized regressor
    fold_ms = []
    for f in range(N_SPLITS):
        tr = fold_id != f
        te = fold_id == f

        mu_by_wk = pd.Series(y_my_all_log[tr]).groupby(pd.Series(wk_my_all[tr])).mean().to_dict()
        sigma_by_wk = pd.Series(y_my_all_log[tr]).groupby(pd.Series(wk_my_all[tr])).std(ddof=0).to_dict()
        fallback_mu = float(np.mean(y_my_all_log[tr]))
        fallback_sigma = float(np.std(y_my_all_log[tr], ddof=0)) or 1.0

        mu_tr = np.array([mu_by_wk.get(k, fallback_mu) for k in wk_my_all[tr]], dtype=float)
        mu_te = np.array([mu_by_wk.get(k, fallback_mu) for k in wk_my_all[te]], dtype=float)
        sigma_tr = np.array([max(float(sigma_by_wk.get(k, fallback_sigma)), 1e-6) for k in wk_my_all[tr]], dtype=float)
        sigma_te = np.array([max(float(sigma_by_wk.get(k, fallback_sigma)), 1e-6) for k in wk_my_all[te]], dtype=float)

        y_std_tr = (y_my_all_log[tr] - mu_tr) / sigma_tr
        reg = fit_lgbm_reg(X_my_all[tr], y_std_tr)
        pred_std_te = reg.predict(X_my_all[te])
        pred_log = pred_std_te * sigma_te + mu_te
        pred = np.expm1(pred_log)
        fold_ms.append(regression_metrics(y_my_all[te], pred, wk_keys=wk_my_all[te]))
    summarize_cv('MySQL workload-standardized regressor', fold_ms)


Within-workload MySQL split:
  test_frac          : 0.2
  train rows         : 1040
  test rows          : 274
  workloads (train)  : 46
  workloads (test)   : 46
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1

[PG-only reg_final (zero-shot)] Within-workload MySQL:
  RMSE                 : 15.9904
  MAPE (%)             : 26.43
  Spearman (pooled)    : 0.4068
  Spearman (wk mean)   : nan  (0/12 valid)
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Aut

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2855: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006088 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13319
[LightGBM] [Info] Number of data points in the train set: 18529, number of used features: 207
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Start training from score 1.554590


...

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [10]:
# Phase 3 - LambdaRank on PostgreSQL
#
# Trained on ALL PG data. Tuned vs v1: min_child_samples, lambda_l1/l2,
# ndcg_eval_at=[1,3,5] for richer diagnostic output.

try:
    from lightgbm import LGBMRanker
except Exception as e:
    raise RuntimeError('LightGBM is required for LambdaRank. pip install lightgbm') from e

order_pg  = np.argsort(wk_pg)
X_pg_s    = X_pg[order_pg]
y_pg_s    = y_pg[order_pg]
wk_pg_s   = wk_pg[order_pg]

rel_pg    = relevance_labels(y_pg_s, wk_pg_s)
groups_pg = group_sizes(wk_pg_s)

ranker = LGBMRanker(
    objective        = 'lambdarank',
    metric           = 'ndcg',
    ndcg_eval_at     = [1, 3, 5],
    n_estimators     = 2000,
    learning_rate    = 0.03,
    num_leaves       = 63,
    min_child_samples= 20,
    lambda_l1        = 0.1,
    lambda_l2        = 0.1,
    subsample        = 0.9,
    colsample_bytree = 0.9,
    random_state     = RANDOM_SEED,
    n_jobs           = -1,
)
ranker.fit(X_pg_s, rel_pg, group=groups_pg)

print('PG ranker trained.  n_estimators =', ranker.n_estimators_)


/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006528 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12366
[LightGBM] [Info] Number of data points in the train set: 20758, number of used features: 156
PG ranker trained.  n_estimators = 2000


In [11]:
# Phase 4 - Transfer to MySQL
#
# IMPROVEMENTS vs v1:
#   1. Held-out MySQL test split (20% of workload groups) - never seen during fine-tuning.
#   2. Pseudo-label quality gate - skip workloads where PG ranker Spearman < PSEUDO_SPEARMAN_GATE.
#   3. Importance weighting - weight PG rows by L2 similarity to MySQL distribution.
#   4. Two-stage fine-tuning:
#      Stage 1: PG (importance-weighted) + gated pseudo-labels at low LR.
#      Stage 2: Real MySQL labels only at even lower LR.
#   5. Zero-shot baseline reported alongside transfer results for fair comparison.

is_my  = (engine == 'mysql').to_numpy()

if not np.any(is_my):
    print('No MySQL rows found; skipping transfer.')
    ranker_my = None
    per_wk_my = per_wk_pg_on_my = {}
    spear_vals, good_wks = [], set()
else:
    X_my  = X[is_my]
    y_my  = y[is_my]
    wk_my = wk[is_my]

    # 4.0  Held-out MySQL test split (by workload group)
    unique_my_wks = np.unique(wk_my)
    n_test_wks    = max(1, int(0.2 * len(unique_my_wks)))
    rng           = np.random.default_rng(RANDOM_SEED)
    test_wks      = set(rng.choice(unique_my_wks, size=n_test_wks, replace=False))
    finetune_mask = np.array([w not in test_wks for w in wk_my])
    test_mask     = ~finetune_mask

    X_my_ft,   y_my_ft,   wk_my_ft   = X_my[finetune_mask],  y_my[finetune_mask],  wk_my[finetune_mask]
    X_my_test, y_my_test, wk_my_test = X_my[test_mask],      y_my[test_mask],      wk_my[test_mask]

    print(f'MySQL fine-tune : {len(X_my_ft)} rows / {len(np.unique(wk_my_ft))} workloads')
    print(f'MySQL held-out  : {len(X_my_test)} rows / {len(np.unique(wk_my_test))} workloads')

    # 4.1  Train Native MySQL Ranker (No PG base margins)
    # The zero-shot PG ranker is feature-blind on MySQL, causing flat predictions and
    # destroying tree margins. We drop `init_model` entirely and learn native MySQL splits.
    
    real_rel_my = relevance_labels(y_my_ft, wk_my_ft)
    order_s2    = np.argsort(wk_my_ft)
    groups_s2   = group_sizes(wk_my_ft[order_s2])
    
    ranker_my = LGBMRanker(
        objective='lambdarank', metric='ndcg', ndcg_eval_at=[1, 3, 5],
        n_estimators=300,         
        learning_rate=0.03,       
        num_leaves=31,            
        min_child_samples=10, 
        reg_alpha=0.1, reg_lambda=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_SEED, n_jobs=-1,
    )
    ranker_my.fit(
        X_my_ft[order_s2], real_rel_my[order_s2],
        group=groups_s2,
    )

    # 4.5  Evaluate on held-out MySQL test set
    score_pg_on_my  = ranker.predict(X_my_test)
    per_wk_pg_on_my = eval_per_workload(y_my_test, -score_pg_on_my, wk_my_test)

    score_my_test = ranker_my.predict(X_my_test)
    per_wk_my     = eval_per_workload(y_my_test, -score_my_test, wk_my_test)

    print('\n-- Held-out MySQL evaluation (never seen during fine-tuning) --')
    print(f'  Zero-shot (PG ranker -> MySQL) : {per_wk_pg_on_my}')
    print(f'  After transfer (Stage 1 + 2)   : {per_wk_my}')

    # 4.6 Adapter Model Transfer (Feature Augmentation)
    if getattr(reg_final, 'booster_', None) is not None:
        print('\n-- Regressor Transfer via PG Adapter Model --')
        
        # 1. Target log & clipping
        y_my_ft_log = np.clip(np.log1p(y_my_ft), -20, 20)
        
        # 2. Extract "behavioral signal" (freeze PG model, predict on MySQL)
        pred_pg_ft = reg_final.predict(X_my_ft)
        pred_pg_test = reg_final.predict(X_my_test)
        
        # 3. Augment MySQL features
        # We append a single scalar column corresponding to PG's expert prediction
        X_my_ft_aug = np.column_stack([X_my_ft, pred_pg_ft])
        X_my_test_aug = np.column_stack([X_my_test, pred_pg_test])
        
        # 4. Train ONLY MySQL adapter model
        # Tuning down n_estimators and leaves to prevent warnings on the small MySQL dataset
        mysql_adapter_model = LGBMRegressor(
            n_estimators=200,
            learning_rate=0.03,
            num_leaves=31,
            min_child_samples=10,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
        
        mysql_adapter_model.fit(X_my_ft_aug, y_my_ft_log)
        
        # 5. Inference pipeline
        pred_adapter_log = mysql_adapter_model.predict(X_my_test_aug)
        pred_adapter = np.expm1(np.clip(pred_adapter_log, -20, 20))
        
        # 6. Evaluate
        rmse_adapter = math.sqrt(mean_squared_error(y_my_test, pred_adapter))
        sp_adapter = spearman_corr(y_my_test, pred_adapter)
        
        print(f'  Adapter Regressor  -> RMSE: {rmse_adapter:.4f} | Spearman: {sp_adapter:.4f}')


MySQL fine-tune : 1091 rows / 37 workloads
MySQL held-out  : 223 rows / 9 workloads
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000490 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1315
[LightGBM] [Info] Number of data points in the train set: 1091, number of used features: 76


/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1

-- Held-out MySQL evaluation (never seen during fine-tuning) --
  Zero-shot (PG ranker -> MySQL) : {'spearman_mean': nan, 'top1_acc': 0.5555555555555556, 'top3_recall': 0.7777777777777778, 'n_

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2855: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/numpy/

In [15]:
# Phase 5 - Save final PG + MySQL models (versioned artifacts)

import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import joblib  # type: ignore
except Exception as e:
    raise RuntimeError("Missing dependency 'joblib'. Install with: pip install joblib") from e

def _as_float(x):
    try:
        return float(x)
    except Exception:
        return None

def _as_str_array(a):
    return np.asarray(a).astype(str)

# --- 1) Save Postgres regressor (log1p target) ---
if 'reg_final' not in globals():
    raise RuntimeError('Expected `reg_final` (PG regressor) to exist. Run Phase 2 first.')

pg_bundle = {
    'model_type': 'pg_regressor',
    'target_transform': 'log1p',
    'invert_transform': 'expm1',
    'plan_repr': PLAN_REPR,
    'feature_names': FEATURE_NAMES,
    'model': reg_final,
}

pg_path = Path(ARTIFACT_DIR) / 'pg_regressor.joblib'
joblib.dump(pg_bundle, pg_path)
print('Saved:', pg_path)

# --- 2) Train+save MySQL workload-standardized regressor ---
if 'X' not in globals() or 'y' not in globals() or 'wk' not in globals() or 'engine' not in globals():
    raise RuntimeError('Expected X/y/wk/engine to exist. Run Phase 1 data prep first.')

engine_arr = engine.to_numpy() if hasattr(engine, 'to_numpy') else np.asarray(engine)
is_my = (engine_arr == 'mysql')
if not np.any(is_my):
    raise RuntimeError('No MySQL rows found in this dataset; cannot train MySQL workload-standardized regressor.')

X_my_all = X[is_my]
y_my_all = y[is_my]
wk_my_all = _as_str_array(wk[is_my])

y_my_all_log = np.log1p(y_my_all)

mu_by_wk = (
    pd.Series(y_my_all_log)
    .groupby(pd.Series(wk_my_all))
    .mean()
    .to_dict()
)
sigma_by_wk = (
    pd.Series(y_my_all_log)
    .groupby(pd.Series(wk_my_all))
    .std(ddof=0)
    .to_dict()
)

fallback_mu = float(np.mean(y_my_all_log))
fallback_sigma = float(np.std(y_my_all_log, ddof=0)) or 1.0

mu_vec = np.array([float(mu_by_wk.get(k, fallback_mu)) for k in wk_my_all], dtype=float)
sigma_vec = np.array([max(float(sigma_by_wk.get(k, fallback_sigma)), 1e-6) for k in wk_my_all], dtype=float)
y_std_all = (y_my_all_log - mu_vec) / sigma_vec

if 'fit_lgbm_reg' not in globals():
    raise RuntimeError('Expected `fit_lgbm_reg` helper to exist. Run Phase 2c first.')

reg_mysql_std = fit_lgbm_reg(X_my_all, y_std_all)

mysql_std_bundle = {
    'model_type': 'mysql_workload_standardized_regressor',
    'target_transform': 'log1p',
    'invert_transform': 'expm1',
    'reconstruction': 'pred_log = pred_std * sigma_wk + mu_wk',
    'plan_repr': PLAN_REPR,
    'feature_names': FEATURE_NAMES,
    'model': reg_mysql_std,
    'mu_by_wk': {str(k): _as_float(v) for k, v in mu_by_wk.items()},
    'sigma_by_wk': {str(k): _as_float(v) for k, v in sigma_by_wk.items()},
    'fallback_mu': fallback_mu,
    'fallback_sigma': fallback_sigma,
}

mysql_path = Path(ARTIFACT_DIR) / 'mysql_workload_standardized_regressor.joblib'
joblib.dump(mysql_std_bundle, mysql_path)
print('Saved:', mysql_path)

meta = {
    'artifact_dir': str(ARTIFACT_DIR),
    'plan_repr': PLAN_REPR,
    'n_features': int(len(FEATURE_NAMES)),
    'pg_model_file': pg_path.name,
    'mysql_model_file': mysql_path.name,
    'n_mysql_workloads': int(pd.Series(wk_my_all).nunique()),
    'n_mysql_rows': int(len(wk_my_all)),
}
meta_path = Path(ARTIFACT_DIR) / 'saved_models_meta.json'
meta_path.write_text(json.dumps(meta, indent=2))
print('Wrote:', meta_path)

Saved: /home/E2ETune-AI4DB/surrogate/artifacts/transfer_rank_surrogate/6ecfae00/pg_regressor.joblib
Saved: /home/E2ETune-AI4DB/surrogate/artifacts/transfer_rank_surrogate/6ecfae00/mysql_workload_standardized_regressor.joblib
Wrote: /home/E2ETune-AI4DB/surrogate/artifacts/transfer_rank_surrogate/6ecfae00/saved_models_meta.json
